# Contest Baseline Workflow Practice

**Topic:** Contest Strategy and Review &nbsp;·&nbsp; **Difficulty:** Medium

## The data (created in the next cell)

- `X_train` `(2500, 15)`, `y_train`: a 3-class task with mild class imbalance and 5 pure-noise features out of 15.
- `X_test` `(900, 15)`: predict these. `y_test` is used by the evaluation cell.

## Your task

1. Record the majority-class baseline, the share of the most common class in your training split (aim to have this within your first minutes).
2. Fit a quick simple model (logistic regression) and note the jump.
3. Make exactly one upgrade (better model family, tuning, or features), verify it on validation, and set `predictions` for `X_test`.

## Evaluation

Test accuracy must reach the target shown in the evaluation cell.


In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(
    n_samples=3400, n_features=15, n_informative=7, n_redundant=3,
    n_classes=3, n_clusters_per_class=2, class_sep=1.3, flip_y=0.03,
    weights=[0.5, 0.3, 0.2], random_state=181,
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=900, random_state=181, stratify=y)
print(f"X_train {X_train.shape}, class balance {[round(float((y_train == c).mean()), 2) for c in (0, 1, 2)]}")


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: run this cell to compare your predictions to the truth
# =====================================================================
import numpy as np

TARGET_ACCURACY = 0.8

if "predictions" not in dir():
    raise NameError("Define `predictions` in the solution cell before running this cell.")

_pred = np.asarray(predictions).ravel()
_true = np.asarray(y_test).ravel()
if _pred.shape != _true.shape:
    raise ValueError(f"`predictions` has shape {_pred.shape}, expected {_true.shape}.")

accuracy = float((_pred == _true).mean())
print("=" * 52)
print(f"Test accuracy : {accuracy:.4f}")
print(f"Target        : >= {TARGET_ACCURACY:.2f}")
print("-" * 52)
print("Per-class recall:")
for _cls in np.unique(_true):
    _mask = _true == _cls
    print(f"  class {_cls!r:>6} : {float((_pred[_mask] == _cls).mean()):.3f}   ({int(_mask.sum())} test samples)")
print("=" * 52)
if accuracy >= TARGET_ACCURACY:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your model and try again.")


## Solution

The time-boxed ladder: majority baseline, quick logistic regression, then exactly one upgrade, a gradient-boosted model, with the two models verified on the same validation split before predicting the test set.

```python
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

Xtr, Xval, ytr, yval = train_test_split(X_train, y_train, test_size=0.2, random_state=0)

majority = float(np.bincount(ytr).max() / len(ytr))
print(f"1) majority baseline : {majority:.3f}")

logreg = LogisticRegression(max_iter=2000).fit(Xtr, ytr)
print(f"2) logistic regression: {logreg.score(Xval, yval):.3f}")

boost = HistGradientBoostingClassifier(random_state=0).fit(Xtr, ytr)
print(f"3) gradient boosting  : {boost.score(Xval, yval):.3f}")

boost.fit(X_train, y_train)
predictions = boost.predict(X_test)
```
